# Notebook 03 — CLIP Feature Labeling
Loads trained SAE checkpoints and labels each feature using CLIP.
Processes a subset of features (first 500 per layer) for speed.

Output: `../checkpoints/labels_layer4.pt`, `../checkpoints/labels_layer12.pt`
Each is a dict: {'labels': [...], 'scores': [...], 'top_k_indices': [...]}

In [1]:
import torch
print(torch.__version__)

2.7.0+cu126


In [2]:
import sys
sys.path.insert(0, '..')
import torch
from torchvision import datasets, transforms
from PIL import Image
from src.sae import SAE
from src.clip_labeler import label_features

DEVICE             = 'cuda'
IMAGENET_VAL_DIR   = 'D:/Master Material/XAI/XAI_project/archive/imagenet-val'  # ← set this
N_IMAGES           = 10000
N_FEATURES         = 500     # label first 500 features per layer
K_PATCHES          = 10
N_SAMPLE_TOKENS    = 50000   # subsample tokens to keep RAM under 2GB

In [3]:
pil_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
dataset = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=None)

token_to_image = {}
for img_idx in range(min(N_IMAGES, len(dataset))):
    img_path = dataset.imgs[img_idx][0]
    img = Image.open(img_path).convert('RGB')
    img = pil_transform(img)
    # map all 197 tokens of this image to the same PIL image
    for tok_offset in range(197):
        token_to_image[img_idx * 197 + tok_offset] = img

print(f'Built token_to_image with {len(token_to_image)} entries')

Built token_to_image with 1970000 entries


In [4]:
def label_layer(acts_path, ckpt_path, save_path, layer_name):
    acts = torch.load(acts_path, weights_only=False)  # (N_tokens, 768) on CPU

    idx = torch.randperm(acts.shape[0], generator=torch.Generator().manual_seed(42))[:N_SAMPLE_TOKENS]
    acts_sample = acts[idx]

    ckpt = torch.load(ckpt_path, weights_only=False)
    sae = SAE(d_input=768, d_hidden=3072).cpu()
    sae.load_state_dict(ckpt['state_dict'])
    sae.eval()

    # 训练时做了中心化+RMS归一化，推断时同样处理
    acts_mean = ckpt['acts_mean']
    acts_rms  = ckpt['acts_rms']
    acts_sample = (acts_sample - acts_mean) / acts_rms

    with torch.no_grad():
        sae_acts, _ = sae(acts_sample)  # (50k, 3072)

    sampled_token_to_image = {
        new_i: token_to_image[idx[new_i].item()]
        for new_i in range(len(idx))
        if idx[new_i].item() in token_to_image
    }

    results = label_features(
        sae_activations=sae_acts[:, :N_FEATURES].to(DEVICE),
        token_to_image=sampled_token_to_image,
        k=K_PATCHES,
        device=DEVICE,
    )
    torch.save(results, save_path)
    print(f'{layer_name}: labeled {len(results["labels"])} features')
    print(f'  sample labels: {results["labels"][:8]}')
    return results

In [5]:
labels4 = label_layer(
    '../data/layer4_activations.pt',
    '../checkpoints/sae_layer4.pt',
    '../checkpoints/labels_layer4.pt',
    'layer4',
)

d:\Tools\Anaconda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
d:\Tools\Anaconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LAN\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warnin

layer4: labeled 500 features
  sample labels: ['wall', 'wall', 'wall', 'wall', 'wall', 'dog', 'bird', 'wall']


In [6]:
labels12 = label_layer(
    '../data/layer12_activations.pt',
    '../checkpoints/sae_layer12.pt',
    '../checkpoints/labels_layer12.pt',
    'layer12',
)

layer12: labeled 500 features
  sample labels: ['wall', 'wall', 'wall', 'bird', 'wall', 'wall', 'wall', 'wall']
